# NudgeOps — Policy Analysis Notebook

Use this notebook to analyze intervention performance, visualize bandit arm states, and run offline policy evaluation.

**Prerequisites:** Backend running at localhost:8000 + demo data seeded.

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime

BASE = 'http://localhost:8000/api/v1'

def get(path, **params):
    r = requests.get(f'{BASE}{path}', params=params)
    r.raise_for_status()
    return r.json()

def post(path, json):
    r = requests.post(f'{BASE}{path}', json=json)
    r.raise_for_status()
    return r.json()

print('Connected to NudgeOps API')
print(requests.get('http://localhost:8000/health').json())

## 1. System Metrics Overview

In [ ]:
metrics = get('/monitoring/metrics')
print(f"Total interventions delivered: {metrics['total_interventions_all_time']}")
print(f"Last 24h deliveries:          {metrics['interventions_last_24h']}")
print(f"Active users (7d):            {metrics['active_users_7d']}")
print(f"Completion rate (7d):         {metrics['completion_rate_7d']:.1%}")
print(f"Negative rate (7d):           {metrics['negative_rate_7d']:.1%}")
print(f"Failing arms:                 {metrics['failing_arms_count']}")

if metrics['alerts']:
    print('\n⚠ ALERTS:')
    for a in metrics['alerts']:
        print(f"  {a['level'].upper()}: {a['message']}")

## 2. Feedback Signal Distribution

In [ ]:
fb = metrics.get('feedback_distribution_7d', {})
if fb:
    signals = list(fb.keys())
    counts = list(fb.values())
    colors = ['#22d3a0','#7c6af7','#9898b8','#f5c542','#f05070']
    
    fig, ax = plt.subplots(figsize=(8, 4))
    fig.patch.set_facecolor('#111118')
    ax.set_facecolor('#111118')
    bars = ax.bar(signals, counts, color=colors[:len(signals)], width=0.5)
    ax.set_xlabel('Feedback Signal', color='#9898b8')
    ax.set_ylabel('Count', color='#9898b8')
    ax.set_title('Feedback Distribution (7d)', color='white', fontsize=13)
    ax.tick_params(colors='#9898b8')
    for spine in ax.spines.values(): spine.set_visible(False)
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                str(count), ha='center', va='bottom', color='white', fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    print('No feedback data yet — seed demo data first')

## 3. Per-User Bandit Arm States

In [ ]:
users = get('/users/')
print(f'Users: {len(users)}')

if users:
    user = users[0]
    print(f'\nAnalyzing: {user["display_name"]} ({user["segment"]})')
    
    states = get(f'/bandit/state/{user["id"]}')
    df = pd.DataFrame(states).sort_values('estimated_success_prob', ascending=False)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor('#111118')
    ax.set_facecolor('#111118')
    
    colors = ['#22d3a0' if p > 0.6 else '#7c6af7' if p > 0.4 else '#f05070' for p in df['estimated_success_prob']]
    bars = ax.barh(df['intervention_type'].str.replace('_', ' '), df['estimated_success_prob'], color=colors)
    ax.axvline(x=0.5, color='#9898b8', linestyle='--', alpha=0.5)
    ax.set_xlabel('Estimated P(success)', color='#9898b8')
    ax.set_title(f'Bandit Arm States — {user["display_name"]}', color='white', fontsize=12)
    ax.tick_params(colors='#9898b8')
    for spine in ax.spines.values(): spine.set_visible(False)
    plt.tight_layout()
    plt.show()
    
    print('\nTop 3 arms:')
    for _, row in df.head(3).iterrows():
        print(f'  {row["intervention_type"]:40s} p={row["estimated_success_prob"]:.3f}  pulls={row["n_pulls"]}')

## 4. Fairness Analysis

In [ ]:
fairness = get('/monitoring/fairness')
print(f'Users analyzed:     {fairness["users_analyzed"]}')
print(f'Fairness cap:       {fairness["fairness_cap"]:.0%}')
print(f'Violations:         {fairness["violation_count"]}')
print(f'Is Fair:            {"✓ Yes" if fairness["is_fair"] else "✗ No"}')

if fairness['violations']:
    print('\nViolations:')
    for v in fairness['violations']:
        print(f'  user={v["user_id"][:12]}  arm={v["intervention_id"][:12]}  share={v["share"]:.0%}')

## 5. Policy Evaluation

In [ ]:
policies = get('/policies/')
active = [p for p in policies if p['status'] == 'active']
if active:
    policy = active[0]
    print(f'Active policy: {policy["name"]} v{policy["version"]} ({policy["bandit_strategy"]})')
    eval_result = get(f'/policies/{policy["id"]}/evaluate', window_days=7)
    if 'error' not in eval_result:
        print(f'\nEvaluation (7d window):')
        print(f'  Deliveries:       {eval_result["total_deliveries"]}')
        print(f'  Response rate:    {eval_result["response_rate"]:.1%}')
        print(f'  Completion rate:  {eval_result["completion_rate"]:.1%}')
        print(f'  Engagement rate:  {eval_result["engagement_rate"]:.1%}')
        print(f'  Mean reward:      {eval_result["mean_reward"]:.4f}')
        print(f'  Health score:     {eval_result["health_score"]}/100')
    else:
        print(f'No data yet: {eval_result["error"]}')
else:
    print('No active policy found')